In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import os
import leidenalg

In [ ]:
import importlib.metadata as md

packages = ["scanpy", "scvi-tools", "numpy", 
            "pandas", "leidenalg", "celltypist", 
            "infercnvpy", "scipy", 'gseapy', 'anndata', 'scikit-learn', 'statsmodels', 'gseapy']

for pkg in packages:
    print(f"{pkg}: {md.version(pkg)}")

import sys
print(sys.version)

In [ ]:
sc.settings.set_figure_params(dpi_save=300)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def set_plot_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "black",
        "grid.color": "#cccccc",
        "grid.linestyle": "--",
        "grid.linewidth": 0.6,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

set_plot_style()

In [ ]:
import gzip
import shutil
import os

for filename in os.listdir('.'):
    if filename.endswith('.gz'):
        unzipped_filename = filename[:-3]
    
        with gzip.open(filename, 'rb') as f_in:
            with open(unzipped_filename, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
                
        print(f"Unzipped: {filename} -> {unzipped_filename}")

In [ ]:
os.makedirs("gz_files", exist_ok=True)

for filename in os.listdir('.'):
    if filename.endswith('.gz'):
        shutil.move(filename, os.path.join("gz_files", filename))
        print(f"Moved: {filename} -> gz_files/")

In [ ]:
features_file = "GSE161529_features.tsv"

all_files = os.listdir('.')

gsm_ids = set(f.split('_')[0] for f in all_files if f.startswith("GSM"))

os.makedirs("Samples", exist_ok=True)

for gsm in gsm_ids:
    sample_folder = os.path.join("Samples", gsm)
    os.makedirs(sample_folder, exist_ok=True)
    
    for f in all_files:
        if f.startswith(gsm):
            if f.endswith("-barcodes.tsv"):
                shutil.move(f, os.path.join(sample_folder, "barcodes.tsv"))
                print(f"Moved {f} → {sample_folder}/barcodes.tsv")
            elif f.endswith("-matrix.mtx"):
                shutil.move(f, os.path.join(sample_folder, "matrix.mtx"))
                print(f"Moved {f} → {sample_folder}/matrix.mtx")
    
    shutil.copy(features_file, os.path.join(sample_folder, "features.tsv"))
    print(f"Copied features.tsv to {sample_folder}")

In [ ]:
ribo_url = "http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt"
ribo_genes = pd.read_table(ribo_url, skiprows=2, header = None)
ribo_genes

In [ ]:
import os
import pandas as pd

base = "Samples"
required = ["matrix.mtx", "features.tsv", "barcodes.tsv"]

rows = []
for sample in sorted(os.listdir(base)):
    folder = os.path.join(base, sample)
    if not os.path.isdir(folder):
        continue

    row = {"sample": sample}
    for f in required:
        if os.path.exists(os.path.join(folder, f)):
            row[f] = "ok"
        elif os.path.exists(os.path.join(folder, f + ".gz")):
            row[f] = "ok (.gz)"
        elif f == "features.tsv" and os.path.exists(os.path.join(folder, "genes.tsv")):
            row[f] = "genes.tsv (old format)"
        else:
            row[f] = "MISSING"
    rows.append(row)

check = pd.DataFrame(rows).set_index("sample")
print(check)

complete = check[~check.isin(["MISSING"]).any(axis=1)]
print(f"\n{len(complete)} of {len(check)} samples complete")

In [ ]:
def pp2(file_path, sample):

    if os.path.exists(os.path.join(file_path, 'matrix.mtx')):
        mtx_file = os.path.join(file_path, 'matrix.mtx')
        features_file = os.path.join(file_path, 'features.tsv')
        barcodes_file = os.path.join(file_path, 'barcodes.tsv')
        gzipped = False
    else:
        print(f"Missing required files in {file_path}, skipping.")
        return None

    try:
        if gzipped:
            adata = sc.read_10x_mtx(file_path)
        else:
            adata = sc.read_mtx(mtx_file).T
            features_df = pd.read_csv(features_file, header=None, sep="\t")
            barcodes_df = pd.read_csv(barcodes_file, header=None)
            adata.var_names = features_df[1].values
            adata.obs_names = barcodes_df[0].values
    except Exception as e:
        print(f"Error loading {sample}: {e}, skipping.")
        return None

    adata.obs_names_make_unique()
    adata.var_names_make_unique()
    adata.obs['Sample'] = sample

    try:
        sc.pp.highly_variable_genes(
            adata,
            n_top_genes=2000,
            subset=True,
            flavor='seurat_v3'
        )
    except Exception as e:
        print(f"HVG failed for {sample}: {e}, skipping sample.")
        return None

    # scVI + SOLO doublet detection
    try:
        scvi.model.SCVI.setup_anndata(adata)
        vae = scvi.model.SCVI(adata)
        vae.train()
        solo = scvi.external.SOLO.from_scvi_model(vae)
        solo.train()
        df = solo.predict()
        df['prediction'] = solo.predict(soft=False)
        df.index = df.index.map(lambda x: x[:-2])
        df['dif'] = df.doublet - df.singlet
        doublets = df[(df.prediction == 'doublet') & (df.dif > 1)]
    except Exception as e:
        print(f"SOLO failed for {sample}: {e}, skipping sample.")
        return None

    # Reload and filter doublets
    try:
        if gzipped:
            adata = sc.read_10x_mtx(file_path)
        else:
            adata = sc.read_mtx(mtx_file).T
            adata.var_names = features_df[1].values
            adata.obs_names = barcodes_df[0].values

        adata.obs_names_make_unique()
        adata.var_names_make_unique()
        adata.obs['Sample'] = sample
        adata.obs['doublet'] = adata.obs.index.isin(doublets.index)
        adata = adata[~adata.obs.doublet]
    except Exception as e:
        print(f"Error filtering doublets for {sample}: {e}, skipping sample.")
        return None

    # QC annotations
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    adata.var['ribo'] = adata.var_names.isin(ribo_genes[0].values)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'],
                               percent_top=None, log1p=False, inplace=True)

    # Filter low-quality cells
    upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
    adata = adata[adata.obs.n_genes_by_counts < upper_lim]
    adata = adata[adata.obs.pct_counts_mt < 15]
    adata = adata[adata.obs.pct_counts_ribo < 20]

    return adata

In [ ]:
samples_dir = 'Samples'
all_files2 = []

for folder in os.listdir(samples_dir):
    folder_path = os.path.join(samples_dir, folder)
    if os.path.isdir(folder_path):
        print(f"Processing {folder}...")
        adata = pp2(folder_path, folder)
        if adata is not None:
            all_files2.append(adata)

print(f"Finished processing {len(all_files2)} datasets.")

In [ ]:
all_files2

In [ ]:
adata = sc.concat(all_files2)

In [ ]:
print(adata)
print(adata.obs["Sample"].value_counts())

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)

In [ ]:
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
from scipy.sparse import csr_matrix

In [ ]:
adata.X = csr_matrix(adata.X)

In [ ]:
ribosomal_genes = [gene for gene in adata.var_names if gene.startswith('RPS') or gene.startswith('RPL')]

In [ ]:
adata = adata[:, ~adata.var_names.isin(ribosomal_genes)]

In [ ]:
print(f"Remaining ribosomal genes: {sum(adata.var_names.str.startswith('RPS')) + sum(adata.var_names.str.startswith('RPL'))}")

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
adata.obs_names = adata.obs["Sample"].astype(str) + "_" + adata.obs_names.astype(str)
adata.var_names_make_unique()
print("unique:", adata.obs_names.is_unique)

In [ ]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer = "counts",
                             categorical_covariate_keys=["Sample"],
                             continuous_covariate_keys=['pct_counts_mt', 'total_counts', 'pct_counts_ribo'])

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model.save('model.model')

In [ ]:
#model = scvi.model.SCVI.load('model.model', adata=adata)

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation()

In [ ]:
adata.layers['scvi_normalized'] = model.get_normalized_expression(library_size = 1e4)

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI')

In [ ]:
sc.tl.leiden(adata, resolution = 0.5)

In [ ]:
sc.tl.umap(adata)

In [ ]:
import celltypist
from celltypist import models

In [ ]:
models.download_models(force_update = True)

In [ ]:
adata.obs_names_make_unique()
adata.var_names_make_unique()

In [ ]:
ct_model = models.Model.load(model = 'Immune_All_Low.pkl')

In [ ]:
predictions = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True)

In [ ]:
adata = predictions.to_adata()

In [ ]:
%pip install GEOparse

In [ ]:
import numpy as np
import pandas as pd
import GEOparse

gse = GEOparse.get_GEO("GSE161529", destdir="./geo", silent=True)

rows = []
for gsm_id, gsm in gse.gsms.items():
    md = gsm.metadata
    row = {"GSM": gsm_id, "title": md["title"][0]}
    for ch in md.get("characteristics_ch1", []):
        k, _, v = ch.partition(":")
        row[k.strip()] = v.strip()
    rows.append(row)

geo_meta = pd.DataFrame(rows).set_index("GSM")

# keep only your 40 samples
my_meta = geo_meta.loc[geo_meta.index.isin(adata.obs["Sample"].unique())].copy()
print(len(my_meta), "of", adata.obs["Sample"].nunique(), "samples matched")

# --- tissue site: primary vs lymph node ---
my_meta["Site"] = np.where(
    my_meta["cell population"].str.contains("lymph", case=False, na=False),
    "Lymph node",
    "Primary",
)

# --- subtype from cancer type (HER2 checked first so it isn't caught by "ER+") ---
ct = my_meta["cancer type"].fillna("")
my_meta["Subtype"] = np.select(
    [
        ct.str.contains("Triple negative", case=False),
        ct.str.contains("HER2", case=False),
        ct.str.contains("ER+", case=False, regex=False),
        ct.str.contains("PR+", case=False, regex=False),
    ],
    ["TNBC", "HER2+", "ER+", "PR+"],
    default="Other",
)

# --- combined label: e.g. "ER+" vs "ER+ LN" ---
my_meta["Group"] = np.where(
    my_meta["Site"] == "Lymph node",
    my_meta["Subtype"] + " LN",
    my_meta["Subtype"],
)

print(my_meta[["title", "patient", "cancer type", "cell population", "Site", "Subtype", "Group"]].to_string())
print(my_meta["Group"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(
    adata, 'leiden',
    method='wilcoxon',
    use_raw=False,
    pts=True
)

markers = sc.get.rank_genes_groups_df(adata, None)
markers = markers[
    (markers.pvals_adj < 0.05) &
    (markers.logfoldchanges > 1) &
    (markers.pct_nz_group > 0.25) &
    (markers.pct_nz_reference < 0.5)
].sort_values(['group','scores'], ascending=[True, False])

In [ ]:
def top_genes_by_group(markers, group, n=50):
    subset = (
        markers[markers["group"].astype(str) == str(group)]
        .sort_values("scores", ascending=False)
        .head(n)
    )

    print(subset["names"].tolist())

    return subset

In [ ]:
top_genes = top_genes_by_group(markers, group="0", n=200)

In [ ]:
##inferCNV

In [ ]:
gene_info = pd.read_csv(f'{path}/hg38__gencode_v27.txt', sep = '\t', header = None)
gene_info.head()

In [ ]:
gene_info.columns = ['gene_symbol', 'chromosome', 'start', 'end']
gene_info.head()

In [ ]:
adata.var.head()

In [ ]:
adata.var['gene_symbol'] = adata_primary.var_names

In [ ]:
adata.var = adata.var.merge(gene_info, on = 'gene_symbol', how = 'left')
adata.var.set_index('gene_symbol', inplace = True)
adata.var

In [ ]:
normal_cell_types = ['G0.Macrophage',
 'G1.T Cell',
 'G11.Endothelial Cell',
 'G15.NK Cell',
 'G16.Macrophage',
 'G17.Myoepithelial Cell',
 'G18.B Cell',
 'G2.Plasma Cell',
 'G21.Mast Cell',
 'G22.Dendritic Cell',
 'G4.Fibroblast']

In [ ]:
cnv.tl.infercnv(
    adata,
    reference_key = 'Cell Type_cnv',
    reference_cat = normal_cell_types,
    window_size = 200,
)

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="Cell Type_cnv")

In [ ]:
cnv.tl.pca(adata)
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata)

In [ ]:
cnv.tl.umap(adata)
cnv.tl.cnv_score(adata)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
wwfig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 11), gridspec_kw={"wspace": 0.5})
ax4.axis("off")
sc.pl.umap(adata, color="cnv_leiden", ax=ax1, show=False)
sc.pl.umap(adata, color="cnv_score", ax=ax2, show=False)
sc.pl.umap(adata, color="Cell Type", ax=ax3)

In [ ]:
adata_epith = adata[adata.obs['Cell Type'] == 'Epithelial']

In [ ]:
print(
    adata_epith.obs.groupby("leiden", observed=True)["cnv_score"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

scores = adata_epith.obs["cnv_score"].dropna().values

kde = gaussian_kde(scores)                     # bw_method="scott" by default
grid = np.linspace(scores.min(), scores.max(), 1000)
density = kde(grid)

peaks, _ = find_peaks(density, prominence=0.05 * density.max())
valleys, _ = find_peaks(-density, prominence=0.05 * density.max())

print("peaks at:", grid[peaks])
print("valleys at:", grid[valleys])

first_valley = None
if len(peaks) > 0:
    after = valleys[valleys > peaks[0]]
    if len(after) > 0:
        first_valley = grid[after[0]]

print("first valley (cutoff):", first_valley)
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(scores, bins=100, density=True, color="lightgrey")
ax.plot(grid, density, color="black", lw=1.5)
ax.plot(grid[peaks], density[peaks], "o", color="tab:blue", label="peaks")
if first_valley is not None:
    ax.axvline(first_valley, color="tab:red", ls="--", label=f"valley = {first_valley:.4f}")
ax.set_xlabel("CNV score")
ax.set_ylabel("Density")
ax.legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()

if first_valley is not None:
    high = (adata_epith.obs["cnv_score"] > first_valley).sum()
    print(f"cells above cutoff: {high} of {adata_epith.n_obs}")